# Gating threshold study: experiments

This notebook finds where to set the welfare gates on the **Balanced hybrid** recommender. It runs on the lab server, which holds
the database and the metro networks, and writes small result files to `results/`. Those files are all
that `analysis.ipynb` needs, so the analysis can run anywhere.

**Gates** (after Uğurel & Yabe 2026)
* **PUP** passes a venue only if $\Pr[U \ge 0] \ge \alpha$. Swept over $\alpha = 0.1, 0.2, \dots, 1.0$.
* **RM** passes a venue only if $\mathbb{E}[\text{regret}] \le \varepsilon$. Swept over $\varepsilon$ set at the 50th–95th percentiles of the expected
  regret of the ungated recommender's suggestions, as in the paper.

Each (city, seed, threshold) gets one PUP run and one RM run. The controls (No RS, Standard RS = ungated Balanced hybrid, and Oracle) run once per
(city, seed) and appear as flat reference lines on the threshold graphs. Size: 8 metros, 2,000 agents, 30 days, 10 seeds.

**Order of work.** The RM grid is defined by Standard RS's regrets, so Standard has to finish before the RM sweep starts.
1. Standard RS and No RS, then calibrate the RM grid from Standard RS.
2. Oracle, the PUP sweep and the RM sweep, all in one pool. Oracle runs are the slowest, so they start first and run alongside the sweeps.
3. Export one CSV per city, then write `results/DONE.txt`.

Each finished simulation is cached under `.cache/gating_study/`. An interrupted notebook (for example after a reboot) resumes from where it
stopped. Run it unattended with `./run_in_background.sh`; progress is appended to `results/progress.log`.

## Setup

In [ ]:
import gc
import json
import math
import multiprocessing as mp
import os
import pickle
import time
from array import array
from dataclasses import dataclass
from datetime import datetime
from itertools import accumulate
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

REPO = Path.cwd().parent

# The server datasource (Postgres: POIs, ACS/PUMS populations, survey personas) is configured in .env.server.
for line in (REPO / ".env.server").read_text().splitlines():
    if line.strip() and not line.lstrip().startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"'))

from welfare_rs import TravelCostEstimator, WelfareAwareOrchestrator, params
from welfare_rs.datasource import get_datasource
from welfare_rs.experiment_harness import table1_metrics
from welfare_rs.metro import build_network
from welfare_rs.oracle_rs import oracle_factory
from welfare_rs.recommender_systems import RECOMMENDER_PRESETS, RecommenderConfig
from welfare_rs.simulation import Simulation
from welfare_rs.utils import haversine_km

In [ ]:
# Experiment size
CITIES = ["houston", "la", "chicago", "sf_bay", "nyc", "dc", "seattle", "miami"]
N_AGENTS = 2000
N_DAYS = 30
SEEDS = list(range(10))

# Threshold grids (the paper's ranges)
PUP_ALPHAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
RM_PERCENTILES = [50, 60, 70, 75, 80, 90, 95]

# Concurrency: up to MAX_PARALLEL runs, one per core, leaving a few cores for the web app and Postgres.
# A run starts only if its memory estimate fits in the budget and the box still has MIN_AVAILABLE_GB free.
# Estimates are measured peaks, with margin, of private memory per run while sharing the network
# (LA, the largest metro: other arms 1.9 GB, Oracle 2.6 GB after one day and growing).
MAX_PARALLEL = 60
MEMORY_BUDGET_GB = 215
RUN_MEMORY_GB = {"oracle": 6.0}
DEFAULT_RUN_MEMORY_GB = 3.0
MIN_AVAILABLE_GB = 25
MAX_ORACLE_PARALLEL = 16  # while other runs wait, so the slow Oracle runs cannot take the whole memory budget

BALANCED_HYBRID = RecommenderConfig(label="Balanced hybrid", **RECOMMENDER_PRESETS["Balanced hybrid"])

RESULTS_DIR = Path("results")
CACHE_DIR = REPO / ".cache" / "gating_study" / f"{N_AGENTS}agents_{N_DAYS}days"
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(CITIES)} cities x {len(SEEDS)} seeds, {N_AGENTS} agents, {N_DAYS} days")

## Two speed-ups

**1. A faster organic venue draw, with identical results.** Profiling showed that about 70% of a simulated day goes to one function. Each
evening, an agent considering a leisure trip computes its straight-line distance to every venue of each subtype. Those distances never
change, because the origin is always the agent's home or work and venues do not move. `FastSimulation` computes them once
per agent and reuses them. The random draw that follows consumes exactly the same random numbers, so every run is bit-for-bit
identical to the engine's. This was verified on NYC and Miami, gated and ungated, by comparing every day's summary and every trip.
Runs become several times faster.

**2. One shared copy of each metro network.** A network is 0.7–3 GB, and it used to be loaded separately by every run. It is
now loaded once here, and every run is a forked copy that shares it. That halves each run's memory, which is what lets one run go on every core.

In [ ]:
class FastSimulation(Simulation):
    # Simulation with the organic venue draw memoised; results are bit-identical to the engine's.
    #
    # Simulation.sample_poi recomputes the straight-line distance from the agent to every venue of a
    # subtype each time it is called (~70% of a simulated day). The origin is always the agent's home or
    # work and venues never move, so the reachable venues and their cumulative weights are computed once
    # per (subtype, origin, reach) and reused. random.choices turns weights into exactly these cumulative
    # weights, so the draw, and every random number after it, is unchanged.

    def __init__(self, *args, **kwargs):
        self._venue_draws = {}
        super().__init__(*args, **kwargs)

    def sample_poi(self, subtype, origin, rng, max_km=None):
        if not self._pois_by_subtype.get(subtype):
            return super().sample_poi(subtype, origin, rng, max_km)  # no venues: engine fallback
        key = (subtype, tuple(origin), max_km)
        if key not in self._venue_draws:
            places = self._pois_by_subtype[subtype]
            if max_km is not None:
                places = [p for p in places if haversine_km(origin, p.location) <= max_km]
            scale = params.CITY_PARAMS.get("organic_poi_proximity_scale_km", 3.0) or 1.0
            weights = [math.exp(-haversine_km(origin, p.location) / scale) for p in places]
            self._venue_draws[key] = (places, array("d", accumulate(weights)), sum(weights))
        places, cum_weights, total = self._venue_draws[key]

        if not places:
            return None, ""
        if len(places) == 1:
            return places[0].location, places[0].place_id
        if total <= 0:
            place = rng.choice(places)
        else:
            place = rng.choices(places, cum_weights=cum_weights, k=1)[0]
        return place.location, place.place_id

In [ ]:
NETWORKS = {city: build_network(city) for city in CITIES}
gc.collect()
gc.freeze()  # keep the garbage collector from touching (and so copying) the shared networks in every run

## Arms and the single-run function

An `Arm` is one recommender condition. Gated arms wrap the Balanced hybrid in the engine's `WelfareAwareOrchestrator`,
wired exactly as the web app wires a gated recommender (`backend/service.py`).

Standard RS goes through the same wrapper in pass-through mode (`"standard"`). It records the expected regret of every candidate,
which is used to calibrate the RM grid. The pass-through filters nothing and draws no random numbers, so the run is identical to an
unwrapped Balanced hybrid.

In [ ]:
@dataclass(frozen=True)
class Arm:
    gate: str                           # "no_rs", "oracle", "standard", "pup" or "rm"
    alpha: Optional[float] = None       # PUP: pass a venue only if P(U >= 0) >= alpha
    epsilon: Optional[float] = None     # RM:  pass a venue only if E[regret] <= epsilon
    percentile: Optional[int] = None    # RM:  the regret percentile epsilon was calibrated at

    @property
    def key(self):
        parts = [self.gate]
        if self.alpha is not None:
            parts.append(f"a{self.alpha:.2f}")
        if self.epsilon is not None:
            parts.append(f"e{self.epsilon:.4f}")
        return "_".join(parts)


class RegretLoggingGate(WelfareAwareOrchestrator):
    # The engine's welfare gate, also recording E[regret] of every candidate it passes.
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.regrets = []

    def recommend(self, user, leisure_subtype, top_k_per_system=5):
        passed = super().recommend(user, leisure_subtype, top_k_per_system)
        self.regrets.extend(rec.expected_regret for recs in passed.values() for rec in recs)
        return passed


def recommender_factory(arm):
    if arm.gate == "no_rs":
        return None
    if arm.gate == "oracle":
        return oracle_factory("Oracle")
    gate = {"mode": "standard" if arm.gate == "standard" else arm.gate}
    if arm.alpha is not None:
        gate["pup_alpha"] = arm.alpha
    if arm.epsilon is not None:
        gate["rm_epsilon"] = arm.epsilon

    def factory(_sim, base_stack):
        return RegretLoggingGate(base_stack, TravelCostEstimator(), **gate)
    return factory


def run_one(city, seed, arm):
    # One simulation. Population and network setup mirror the web app's study runner.
    ds = get_datasource()
    population = ds.population(city, replicate=seed % ds.population_replicates(city),
                               limit=N_AGENTS + max(100, N_AGENTS // 5))
    if population is None or len(population) < N_AGENTS:
        raise RuntimeError(f"{city}: no built population with {N_AGENTS} agents")
    sim = FastSimulation(
        num_agents=N_AGENTS, seed=seed,
        use_recommenders=(arm.gate != "no_rs"),
        persona_rows=ds.personas(),
        persona_csv_path=ds.persona_csv_path() or params.SURVEY_PERSONA_CSV_PATH,
        population_rows=population,
        road_network=NETWORKS[city],
        poi_rows=ds.poi_rows(city),
        disabled_modes=("transit",),
        recommender_config=BALANCED_HYBRID,
        recommender_factory=recommender_factory(arm),
    )
    sim.run_days(N_DAYS)
    metrics = table1_metrics(sim)  # paper Table 1, over the evaluation (last) day's leisure trips
    return {
        "city": city, "seed": seed, "arm": arm.key, "gate": arm.gate,
        "alpha": arm.alpha, "percentile": arm.percentile, "epsilon": arm.epsilon,
        "mean_utility": metrics["mean_utility"], "abstention_rate": metrics["abstention_rate"],
        "regrets": np.asarray(sim.recommender_stack.regrets, dtype=np.float32) if arm.gate == "standard" else None,
    }

## Parallel runner

Each `(city, seed, arm)` simulation runs in its own process, forked from this notebook so that it shares the networks. When a run finishes,
its process exits and the memory goes back to the system. A run is started whenever a core is free and its memory estimate fits.
A failed run is logged and does not stop the others; re-running the notebook retries only the runs that are missing.

In [ ]:
def log(message):
    line = f"{datetime.now():%Y-%m-%d %H:%M}  {message}"
    print(line, flush=True)
    with open(RESULTS_DIR / "progress.log", "a") as f:
        f.write(line + "\n")


def available_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6


def run_memory_gb(task):
    return RUN_MEMORY_GB.get(task[2].gate, DEFAULT_RUN_MEMORY_GB)


def can_start(task, running, pending):
    free_gb = MEMORY_BUDGET_GB - sum(run_memory_gb(other) for other in running.values())
    if run_memory_gb(task) > free_gb:
        return False
    if task[2].gate != "oracle" or all(other[2].gate == "oracle" for other in pending):
        return True  # the Oracle cap only applies while other runs are waiting
    return sum(other[2].gate == "oracle" for other in running.values()) < MAX_ORACLE_PARALLEL


def cache_path(city, seed, arm):
    return CACHE_DIR / f"{city}_seed{seed}_{arm.key}.pkl"


def run_grid(arms):
    # Run every (city, seed, arm) not yet cached, in the order of `arms`, and return all of them as one row per run.
    tasks = [(city, seed, arm) for arm in arms for city in CITIES for seed in SEEDS]
    pending = [task for task in tasks if not cache_path(*task).exists()]
    n_todo = len(pending)
    log(f"{len(tasks) - n_todo} of {len(tasks)} runs cached; running {n_todo}")

    running, failed, finished = {}, [], 0
    # maxtasksperchild=1: a fresh process per run. fork: runs share the networks and the functions defined here.
    with mp.get_context("fork").Pool(MAX_PARALLEL, maxtasksperchild=1) as pool:
        while pending or running:
            # Start the first pending runs that fit, while cores and memory allow.
            while pending and len(running) < MAX_PARALLEL and available_gb() >= MIN_AVAILABLE_GB:
                task = next((task for task in pending if can_start(task, running, pending)), None)
                if task is None:
                    break
                pending.remove(task)
                running[pool.apply_async(run_one, task)] = task
            time.sleep(10)
            for job in [job for job in running if job.ready()]:
                city, seed, arm = task = running.pop(job)
                finished += 1
                try:
                    result = job.get()
                except Exception as exc:
                    failed.append(task)
                    log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  FAILED: {exc!r}")
                    continue
                path = cache_path(*task)
                tmp = path.with_suffix(".tmp")
                tmp.write_bytes(pickle.dumps(result))
                tmp.replace(path)  # atomic: an interrupted write never leaves a half-written cache file
                log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  U = {result['mean_utility']:+.3f}  "
                    f"abstention = {result['abstention_rate']:.2f}")
    if failed:
        raise RuntimeError(f"{len(failed)} runs failed (see results/progress.log); re-run to retry them")
    return pd.DataFrame([pickle.loads(cache_path(*task).read_bytes()) for task in tasks])

## Phase 1: Standard RS and No RS

In [ ]:
NO_RS, STANDARD, ORACLE = Arm("no_rs"), Arm("standard"), Arm("oracle")
phase1 = run_grid([STANDARD, NO_RS])

## RM calibration

RM thresholds have no natural scale, so the paper sets $\varepsilon$ at percentiles of the expected regret that Standard RS's suggestions carry. The percentiles
are taken over every candidate it recommended, in every city, seed and day, which gives one $\varepsilon$ grid shared by all cities. A lower percentile
is a stricter gate.

In [ ]:
regrets = np.concatenate(phase1.loc[phase1["gate"] == "standard", "regrets"].to_list())
RM_EPSILONS = {p: round(float(np.percentile(regrets, p)), 4) for p in RM_PERCENTILES}
(RESULTS_DIR / "rm_epsilons.json").write_text(json.dumps(
    {"n_recommendations": len(regrets), "epsilon_by_percentile": RM_EPSILONS}, indent=2))

print(f"{len(regrets):,} Standard-RS recommendations")
pd.Series(RM_EPSILONS, name="epsilon").rename_axis("percentile").to_frame()

## Phase 2: Oracle and the PUP and RM sweeps

Oracle runs go first because they are the longest: Oracle ranks every venue by the agent's true utility. The sweep runs fill the remaining cores
around them, and once the sweeps are done the remaining Oracle runs can use every core.

In [ ]:
PUP_ARMS = [Arm("pup", alpha=alpha) for alpha in PUP_ALPHAS]
RM_ARMS = [Arm("rm", epsilon=eps, percentile=p) for p, eps in RM_EPSILONS.items()]
phase2 = run_grid([ORACLE, *PUP_ARMS, *RM_ARMS])

## Export the results

One CSV per city with one row per run, holding only what the threshold graphs need: the arm and its threshold, plus the run's
evaluation-day mean net utility and abstention rate (paper Table 1 definitions). `config.json` records the design. `DONE.txt` is written
last, so it appears only once every result file is complete.

In [ ]:
runs = pd.concat([phase1, phase2], ignore_index=True)
columns = ["city", "seed", "arm", "gate", "alpha", "percentile", "epsilon", "mean_utility", "abstention_rate"]
for city, city_runs in runs.groupby("city"):
    city_runs = city_runs[columns].astype({"percentile": "Int64"})  # integer percentiles, blank where not RM
    city_runs.sort_values(["arm", "seed"]).to_csv(RESULTS_DIR / f"{city}.csv", index=False)

(RESULTS_DIR / "config.json").write_text(json.dumps({
    "cities": CITIES, "n_agents": N_AGENTS, "n_days": N_DAYS, "seeds": SEEDS,
    "recommender": "Balanced hybrid", "pup_alphas": PUP_ALPHAS, "rm_percentiles": RM_PERCENTILES,
}, indent=2))

(RESULTS_DIR / "DONE.txt").write_text(
    f"All experiments finished {datetime.now():%Y-%m-%d %H:%M}.\n"
    f"{len(runs)} runs: {len(CITIES)} cities x {len(SEEDS)} seeds x {runs['arm'].nunique()} arms, "
    f"{N_AGENTS} agents, {N_DAYS} days.\n"
)
log(f"DONE: {len(runs)} runs exported to {RESULTS_DIR.resolve()}")